# Project 31 — LangGraph Human Approval Workflow
## Agent Pauses for Human Approval Before Executing Actions

**Stack:** LangGraph · Ollama · Jupyter

In [ ]:
# !pip install -q langgraph langchain langchain-ollama

## Step 1 — Setup

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.1)

class WorkflowState(TypedDict):
    request: str
    analysis: str
    proposed_action: str
    approval_status: str  # pending, approved, rejected
    result: str

## Step 2 — Define Workflow Nodes

In [ ]:
def analyze_request(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Analyze this request and determine what action is needed. "
         "Classify risk as LOW, MEDIUM, or HIGH."),
        ("human", "{request}")
    ])
    chain = prompt | llm | StrOutputParser()
    analysis = chain.invoke({"request": state["request"]})
    print(f"  📋 Analysis complete")
    return {"analysis": analysis}

def propose_action(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Based on the analysis, propose a specific action to take. "
         "Be concrete about what will happen."),
        ("human", "Request: {request}\nAnalysis: {analysis}")
    ])
    chain = prompt | llm | StrOutputParser()
    action = chain.invoke({"request": state["request"], "analysis": state["analysis"]})
    print(f"  🎯 Proposed action ready")
    return {"proposed_action": action, "approval_status": "pending"}

def simulate_human_approval(state: WorkflowState) -> WorkflowState:
    """In a real app, this would pause and wait for human input.
    Here we simulate approval based on risk level."""
    analysis = state.get("analysis", "")
    # Auto-approve LOW risk, prompt for MEDIUM/HIGH
    if "LOW" in analysis.upper():
        print(f"  ✅ Auto-approved (low risk)")
        return {"approval_status": "approved"}
    elif "HIGH" in analysis.upper():
        print(f"  ❌ Rejected (high risk — requires manual review)")
        return {"approval_status": "rejected"}
    else:
        print(f"  ✅ Approved (medium risk)")
        return {"approval_status": "approved"}

def execute_action(state: WorkflowState) -> WorkflowState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Execute the approved action and report the result."),
        ("human", "Action: {action}")
    ])
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({"action": state["proposed_action"]})
    print(f"  🚀 Action executed")
    return {"result": result}

def reject_action(state: WorkflowState) -> WorkflowState:
    return {"result": f"Action REJECTED. Proposed action was: {state['proposed_action'][:100]}..."}

def route_approval(state: WorkflowState) -> Literal["execute", "reject"]:
    return "execute" if state["approval_status"] == "approved" else "reject"

## Step 3 — Build the Graph

In [ ]:
graph = StateGraph(WorkflowState)

graph.add_node("analyze", analyze_request)
graph.add_node("propose", propose_action)
graph.add_node("approve", simulate_human_approval)
graph.add_node("execute", execute_action)
graph.add_node("reject", reject_action)

graph.set_entry_point("analyze")
graph.add_edge("analyze", "propose")
graph.add_edge("propose", "approve")
graph.add_conditional_edges("approve", route_approval, {
    "execute": "execute",
    "reject": "reject",
})
graph.add_edge("execute", END)
graph.add_edge("reject", END)

app = graph.compile()
print("Human approval workflow compiled!")

## Step 4 — Test With Different Risk Levels

In [ ]:
requests = [
    "Update the user's email address in the system",          # Low risk
    "Deploy the new feature to production servers",           # Medium risk
    "Delete all user data for GDPR compliance request",       # High risk
]

for req in requests:
    print(f"\n{'='*60}")
    print(f"Request: {req}")
    print("-"*60)
    result = app.invoke({
        "request": req, "analysis": "", "proposed_action": "",
        "approval_status": "pending", "result": ""
    })
    print(f"\nFinal Result: {result['result'][:200]}")

## What You Learned
- **Human-in-the-loop** approval gates in LangGraph
- **Risk-based routing** with conditional edges
- **Workflow state management** across nodes